In [25]:
# Find_Select_Cemeteries.py
# 9/26/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep_new(get_counter):
    global sleep_time_pause,sleep_range,pause_count
    get_counter+=1
    if get_counter%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print('\n')
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds   Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    time.sleep(sleep_time)    
    return get_counter

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

# def get_grave_count(url):
#     global status_code
#     capture_json=[]
#     grave_count=0
#     response = requests.request("GET", url, headers=my_header) 
#     status_code=response.status_code
#     url_source = response.text 
#     if status_code!=200:print('\tstatus_code:',status_code, 'url',url);return capture_json,grave_count
#     beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
#     specific_tag_text = beautifulSoupText.get_text()
#     capture_json=json.loads(specific_tag_text) 
#     grave_count=capture_json['total']
#     return capture_json,grave_count
    
# def extract_grave_data(capture_json):
#     global field_list_set,field_list
#     collection_df=pd.json_normalize(capture_json['collection'])
#     collection_columns_set=set(list(collection_df))
#     missing_columns=field_list_set-collection_columns_set
#     collection_df[list(missing_columns)]=np.nan
#     cemetery_item_df=collection_df[field_list]
#     return cemetery_item_df

def clean_dataframe(cemetery_df):
    global field_list
    for column in field_list:
        cemetery_df[column]= cemetery_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        cemetery_df[column]= cemetery_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            cemetery_df[column]= cemetery_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return cemetery_df

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    for iTry in range(0,2):
        response = requests.request("GET", url, headers=my_header) 
        status_code=response.status_code
        url_source = response.text 
        if status_code!=200:
            print('\tstatus_code:',status_code, 'url',url)
            time.sleep(60)
            continue
        beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
        specific_tag_text = beautifulSoupText.get_text()
        capture_json=json.loads(specific_tag_text) 
        try:
            grave_count=capture_json['total']
        except:
            specific_tag_text
            print(capture_json)
            print(f" XXXX get_grave_count Failed: Try: {iTry} \n url: {url}")
            time.sleep(60) 
    return capture_json,grave_count

def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    try:
        collection_df['links']=collection_df['links'].str[0].str[2]
    except:
        collection_df['links']=np.nan
    cemetery_item_df=collection_df[field_list]
    return cemetery_item_df


#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************
my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Search_All_Cemeteries '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileIn="Missing_Cemeteries_Short_X"
fileOutput="Missing_Cemeteries_Short_Output"

fileCemeteriesIn=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+fileIn+".csv"
fileOut=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+fileOutput+"_"+timestamp+".csv"

fileOut0=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+fileOutput+".csv"
fileOutInt=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+fileOutput+"_Interim.csv"

cemetery_id_df=pd.read_csv(fileCemeteriesIn, encoding='latin1')


field_list=['cemeteryName','cemeteryCityName','cemeteryStateName','cemeteryCountyName','cemeteryCountyId','cemeteryStateId']
field_list_set=set(field_list)

total_cemetery_count=len(cemetery_id_df)
sleep_range=[5,10]
sleep_time_pause=1*60
pause_count=100
get_counter=0
cemetery_count=0
cemetery_df=pd.DataFrame([])

url0 = "https://www.findagrave.com/cemetery/"
url1 ="/memorial-search?ajax=true&skip=&limit=100&firstname=&middlename=&lastname=&birthyear=1925&birthyearfilter=before"
url2a ="&datefilter=&orderby=b"

for index, row in cemetery_id_df.iterrows():
    cemetery_id=row['cemetery_id']
    cemetery_count+=1
    url=url0+str(cemetery_id)+url1+url2a
    capture_json,grave_count=get_grave_count(url)
    if status_code!=200:print('\tstatus_code:',status_code, 'frame count:',frame_count);break_flag=1;break 

    cemetery_item_df=extract_grave_data(capture_json)
    cemetery_item_df['cemetery_id']=cemetery_id
    cemetery_item_df['memorial_count']=grave_count
    cemetery_item_df['prob_jewish']=0
    cemetery_item_df['prob_jewish_trunc']=0
    cemetery_item_df['low_grave_count']=0

    cemetery_df = pd.concat([cemetery_df,cemetery_item_df]) 
    cemetery_df = cemetery_df.drop_duplicates() 
    cemetery_df.reset_index(drop=True, inplace = True)   
    cemetery_df=clean_dataframe(cemetery_df)
    cemetery_name=''
    if len(cemetery_item_df['cemeteryName'])>0:cemetery_name=cemetery_item_df['cemeteryName'].iloc[0]
    
    cemetery_df.to_csv(fileOutInt,mode="w", index=False,header=True)
    print(f"    --- {process_name:<20}• Cemetery: {cemetery_name} • {cemetery_id} ••  • grave count: {grave_count} Cemetery Count: {cemetery_count}/{total_cemetery_count} ")
    get_counter=random_sleep_new(get_counter) 

reorder_list=['cemeteryStateName','cemeteryCityName','cemeteryName','prob_jewish','prob_jewish_trunc','low_grave_count','cemetery_id','memorial_count','cemeteryCountyName','cemeteryCountyId','cemeteryStateId']
rename_list=['cemetery_state','cemetery_city','cemetery_name','prob_jewish','prob_jewish_trunc','low_grave_count','cemetery_id','memorial_count','cemetery_county','cemetery_county_id','cemetery_state_id']

cemetery_df=cemetery_df[reorder_list]
cemetery_df.columns=rename_list

cemetery_df = cemetery_df.drop_duplicates()        
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   
# cemetery_df=clean_dataframe(cemetery_df)  
cemetery_df.to_csv(fileOut,mode="w", index=False,header=True)
cemetery_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


**Starting ***  Search_All_Cemeteries   *** , Time: 2025-09-26 16:46:52.413628  **

    --- Search_All_Cemeteries • Cemetery:  • 2678332 ••  • grave count: 0 Cemetery Count: 1/53 
    --- Search_All_Cemeteries • Cemetery: Holy Ghost Fathers Cemetery • 2679559 ••  • grave count: 35 Cemetery Count: 2/53 
    --- Search_All_Cemeteries • Cemetery: First United Methodist Church Memorial Garden • 2680000 ••  • grave count: 47 Cemetery Count: 3/53 
    --- Search_All_Cemeteries • Cemetery: Our Saviour's Lutheran Church Memorial Garden • 2680146 ••  • grave count: 111 Cemetery Count: 4/53 
    --- Search_All_Cemeteries • Cemetery: Saint Matthew's Episcopal Church Memorial Garden • 2694173 ••  • grave count: 62 Cemetery Count: 5/53 
    --- Search_All_Cemeteries • Cemetery: First United Methodist Church of Mesa Columbarium • 2694489 ••  • grave count: 37 Cemetery Count: 6/53 
    --- Search_All_Cemeteries • Cemetery: Desert Cross Lutheran Church Columbarium • 2694535 ••  • grave count: 5 Cemete

KeyError: 'cemeteryName'

In [24]:
# if cemetery_item_df['cemeteryName'].not.empty:print('Bad')


0